In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer

# Below libraries are for similarity matrices using sklearn
from sklearn.metrics.pairwise import cosine_similarity  
from sklearn.metrics import pairwise_distances

import fasttext
from huggingface_hub import hf_hub_download
import random

In [2]:
data = pd.read_csv('Dataset/News_Category_Dataset_v3.csv', index_col=False)

In [3]:
headline_vectorizer = CountVectorizer()
headline_features   = headline_vectorizer.fit_transform(data['headline'])

In [4]:
model_path = hf_hub_download(repo_id="facebook/fasttext-en-vectors", filename="model.bin")
model = fasttext.load_model(model_path)
vocabulary = model.words

In [5]:
headline_encoded = []
for i in data['headline']:
    fasttext_word = np.zeros(300, dtype="float32")
    for word in i.split():
        if word in vocabulary:
            fasttext_word = np.add(fasttext_word, model[word])
    fasttext_word = np.divide(fasttext_word, len(i.split()))
    headline_encoded.append(fasttext_word)

headline_encoded = np.array(headline_encoded)

In [6]:
def avg_w2v_with_category_authors_and_publshing_day(row_index, num_similar_items, cat_list): #headline_preference = True, category_preference = False):
    w2v_dist  = pairwise_distances(headline_encoded, headline_encoded[row_index].reshape(1,-1))
    indices = np.argsort(w2v_dist.flatten())[0:num_similar_items].tolist()
    df = pd.DataFrame({
                'headline_text': data['headline'][indices].values,
                'Word2Vec based Euclidean similarity': w2v_dist[indices].ravel(),
                'Category': data['category'][indices].values,
                'Authors': data['authors'][indices].values,
                'Day and month': data['day and month'][indices].values})
    # df = df[df['Category'].isin(cat_list)]
    print("="*30,"Queried article details","="*30)
    print('headline : ',data['headline'][indices[0]])
    print('Category : ', data['category'][indices[0]])
    print('Authors : ', data['authors'][indices[0]])
    print('Day and month : ', data['day and month'][indices[0]])
    print("\n","="*25,"Recommended articles : ","="*23)
    #return df.iloc[1:,[1,7,8,9]]
    return df.iloc[1:, ]


avg_w2v_with_category_authors_and_publshing_day(210,10,['ARTS & CULTURE'])

============================== Queried article details ==============================
headline :  andy richters parent tweet hilarious
Category :  PARENTS
Authors :  Caroline Bologna
Day and month :  Mon_Jan

 ========================= Recommended articles :  =======================


,headline_text,Word2Vec based Euclidean similarity,Category,Authors,Day and month
1,apple cofounder steve wozniak ditch facebook d...,0.477018,BUSINESS,Ed Mazza,Mon_Apr
2,scientists celebrities share emotional tribute...,0.485799,SCIENCE,Ed Mazza,Wed_Mar
3,twitter ceo jack dorsey call trump thoughts pr...,0.487337,MEDIA,Ed Mazza,Wed_Apr
4,jimmy kimmel ask people take side donald trump...,0.499276,COMEDY,Lee Moran,Tue_Jan
5,lindsey jacobellis suffer winter olympics hear...,0.499453,SPORTS,Ron Dicker,Fri_Feb
6,olympic sprint king lamont marcell jacobs reco...,0.499562,SPORTS,Ron Dicker,Mon_Aug
7,stephen colbert troll old white guy bernie san...,0.500349,COMEDY,Ed Mazza,Wed_Feb
8,dave chappelle shred white house take offense ...,0.501275,COMEDY,Ed Mazza,Tue_May
9,lovely quote parenthood john legend,0.504061,PARENTING,Caroline Bologna,Sat_Dec


In [10]:
class Recommender:
    def __init__(self):
        self.data = pd.read_csv('Dataset/News_Category_Dataset_v3.csv', index_col=False)
        headline_vectorizer = CountVectorizer()
        headline_features = headline_vectorizer.fit_transform(self.data['headline'])
        model_path = hf_hub_download(repo_id="facebook/fasttext-en-vectors", filename="model.bin")
        self.model = fasttext.load_model(model_path)
        self.vocabulary = self.model.words
        headline_encoded = []
        for i in data['headline']:
            fasttext_word = np.zeros(300, dtype="float32")
            for word in i.split():
                if word in self.vocabulary:
                    fasttext_word = np.add(fasttext_word, self.model[word])
            fasttext_word = np.divide(fasttext_word, len(i.split()))
            headline_encoded.append(fasttext_word)

        self.headline_encoded = np.array(headline_encoded)
    
    def reset():
        self.data = pd.read_csv('Dataset/News_Category_Dataset_v3.csv', index_col=False)
        headline_vectorizer = CountVectorizer()
        headline_features = headline_vectorizer.fit_transform(self.data['headline'])
        headline_encoded = []
        for i in data['headline']:
            fasttext_word = np.zeros(300, dtype="float32")
            for word in i.split():
                if word in self.vocabulary:
                    fasttext_word = np.add(fasttext_word, self.model[word])
            fasttext_word = np.divide(fasttext_word, len(i.split()))
            headline_encoded.append(fasttext_word)
        self.headline_encoded = np.array(headline_encoded)

    def recommend(self, read_ids, cat_list):
        # row_index, num_similar_items,
        if(len(read_ids) == 0):
            rec = self.data['id'].tolist()
            random.shuffle(rec)
            return rec[:10]

        titles = data.loc[data['id'].isin(read_ids), 'headline'].tolist()
        titles_encoded = []
        for i in titles:
            fasttext_word = np.zeros(300, dtype="float32")
            for word in i.split():
                if word in self.vocabulary:
                    fasttext_word = np.add(fasttext_word, self.model[word])
            fasttext_word = np.divide(fasttext_word, len(i.split()))
            titles_encoded.append(fasttext_word)

        titles_encoded = np.array(titles_encoded)

        new_df = []
        for i in titles_encoded:
            fasttext_dist  = pairwise_distances(self.headline_encoded, i.reshape(1,-1))
            indices = np.argsort(fasttext_dist.flatten())[0:200].tolist()
            if(len(new_df) == 0):
                new_df = pd.DataFrame({
                            'headline_text': self.data['headline'][indices].values,
                            'id': self.data['id'][indices].values,
                            'Category': self.data['category'][indices].values})
            else:
                new_df = pd.concat([new_df, (pd.DataFrame({
                            'headline_text': self.data['headline'][indices].values,
                            'id': self.data['id'][indices].values,
                            'Category': self.data['category'][indices].values}))], axis=0)
        new_df = new_df.iloc[1:, ]
        if(len(cat_list) == 0):
            rec = new_df['id'].tolist()
            random.shuffle(rec)
            return rec[:10]
        new_df = new_df[new_df['Category'].isin(cat_list)]
        
        # print(titles)
        # print(new_df.to_string())
        return new_df['id'].tolist()[:10]

In [11]:
model = Recommender()

In [12]:
model.recommend([10028, 123], ['WORLD', 'POLITICS', 'SPORTS'])

[14166, 2246, 14114, 11740, 11454, 14723, 13143, 13752, 1990, 3183]